# Simonyi dome open/close retrospective (RSO-1175 / RSO-1176)

Replays the written weather guidance against EFD telemetry, minute by minute, to answer two questions:

1. **One night (default dayObs 20261028):** when would each version of the rules have allowed the dome to open, did any humidity spike reach a closing limit, and what did the logs, Watcher and scheduler record?
2. **Commissioning through operations:** per night, the hours the guidance allowed the dome to be open versus the hours it was actually open. This flags nights that may have been missed (closed but allowed) and risky nights (open past a closing limit).

Guidance replayed (Confluence OOD space):
* *Weather Constraints* (page 39696225, v36, 2026-10-27): summary table (dewpoint spread governs, humidity de-prioritized) **and** the "Deciding to open" prose (RH > 70% = do not open).
* *Recovering from precipitation* (page 1241514042, v10, draft): RH > 75% keeps the dome closed; 20-30 min of safe conditions before opening.
* A **proposed** rule: open when the rolling 30-min *median* spread is > 5 °C and the 30-min *minimum* is > 2 °C. A short spike does not restart the clock.

Run this on the USDF RSP (Nublado) or Times Square. Data access goes through `rubin_nights`, the same as the other notebooks in this repo.
Cloud cover has no direct EFD measurement. The allskycam cloud index (rubin_allskycam products under `ALLSKY_OUT`) is used where available, plus DIMM/RINGSS measurement cadence as a lower-bound proxy for clear sky (neither has published since mid-2026); review the all-sky movie for the night by eye.

*AI-assisted (NOIRL-POL-006): drafted with Claude, to be reviewed before use.*

In [ ]:
# Parameters
DAY_OBS = 20261028          # night to replay in detail
SEASON_MIN = 20250415       # start of the season-wide scan (LSSTCam on sky)
SEASON_MAX = 20261101       # end of the season-wide scan
RUN_SEASON = False          # the season scan makes many EFD queries; turn on when wanted
SEASON_BIN = "5m"           # aggregation for the season scan
OUTSIDE_INDEX = 301         # ESS:301 = outside weather station
CAMERA_DB = "lsst.MTCamera" # MTCamera influx database (not the EFD)
DYN_TOPIC = "lsst.MTCamera.utiltrunk_ut"
DYN_FIELD = "CoolPipeSplyTemp"  # Dynalene supply measured at the camera (on the telescope)
DYN_OFFSET = 0.0            # added to DYN_FIELD; the camera reading is already the on-telescope temperature
                            # (source setpoint -2 C, warms to ~2 C at the TMA supply and ~3.5 C at the camera)
TOPEND_INDEX = 111          # ESS:111 = top end (dew point recorded on closure per the guidance page)
SITE_TZ = "America/Santiago"

In [ ]:
try:
    import rubin_nights  # noqa: F401
except ImportError:
    !pip install --user --upgrade git+https://github.com/lsst-sims/rubin_nights.git --no-deps > /dev/null 2>&1

import re
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from IPython.display import display
from astropy.time import Time, TimeDelta

from rubin_nights import connections
import rubin_nights.dayobs_utils as rn_dayobs
import rubin_nights.observatory_status as observatory_status

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_colwidth", 200)
pd.set_option("display.width", 200)

from rubin_nights.influx_query import InfluxQueryClient

endpoints = connections.get_clients()
efd = endpoints["efd"]

# The camera writes utiltrunk telemetry to its own influx database, reached with the same RSP token.
try:
    cam = InfluxQueryClient(efd.site, db_name=CAMERA_DB, repertoire_site=efd.repertoire_site,
                            auth=("user", connections.get_access_token()))
except Exception as e:  # noqa: BLE001
    print(f"camera influx via repertoire failed ({e}); trying segwarides")
    cam = InfluxQueryClient(efd.site, db_name=CAMERA_DB)

## Guidance thresholds

All limits are expressed as the ambient dewpoint spread (T − T_dew, °C) where possible. The page's own formula, T_dew = T − (100 − RH)/5, gives RH 70% → 6 °C, 75% → 5 °C, 80% → 4 °C and 90% → 2 °C.

In [ ]:
G = dict(
    spread_open=5.0,        # table: can open above this
    spread_close=2.0,       # table: close below this (ambient, instantaneous)
    sensor_close=2.0,       # any sensor within 2 C of its dew point ...
    sensor_close_min=20,    # ... for ~20 min -> close
    rh_prose=70.0,          # "Deciding to open" prose: RH > 70% -> do not open
    rh_precip=75.0,         # precipitation proposal: RH > 75% -> keep closed
    wind_avg_close=20.0,    # 5-min average wind, m/s
    gust_close=25.0,        # gusts, m/s (multiple points)
    reopen_stable_min=30,   # "conditions stable for 30 min"
    before_sunrise_h=1.0,   # "> 1 hour before sunrise"
    spike_rh=80.0,          # what the night report called a spike
)
G

## Helpers

In [ ]:
def _iso(t):
    return f"{Time(t).utc.isot}Z"


def efd_agg(topic, field, t0, t1, every="1m", agg="mean", index=None, client=None):
    # Aggregated influx series (UTC index); `client` defaults to the EFD. Empty Series if the topic or field is missing.
    where = f"time >= '{_iso(t0)}' AND time <= '{_iso(t1)}'"
    if index is not None:
        where += f" AND salIndex = {int(index)}"
    q = f'SELECT {agg}("{field}") AS v FROM "{topic}" WHERE {where} GROUP BY time({every}) fill(none)'
    df = (client or efd).query(q)
    if df is None or len(df) == 0 or "v" not in df:
        return pd.Series(dtype=float, name=field)
    return df["v"].astype(float).rename(field)


def efd_raw(topic, fields, t0, t1, index=None):
    try:
        df = efd.select_time_series(topic, fields, Time(t0), Time(t1), index=index)
    except Exception as e:  # noqa: BLE001
        print(f"{topic}: {type(e).__name__}: {e}")
        return pd.DataFrame()
    return df if df is not None else pd.DataFrame()


def first_field(topic, prefix):
    # The first field of `topic` whose name starts with `prefix` (handles array fields such as temperatureItem0).
    try:
        f = efd.get_fields(topic)
    except Exception:  # noqa: BLE001
        return None
    col = "fieldKey" if "fieldKey" in f.columns else f.columns[0]
    names = sorted(n for n in f[col].astype(str) if n.startswith(prefix))
    return names[0] if names else None


def outside_weather(t0, t1, every="1m", index=OUTSIDE_INDEX):
    t_field = first_field("lsst.sal.ESS.temperature", "temperatureItem") or "temperatureItem0"
    cols = {
        "T": efd_agg("lsst.sal.ESS.temperature", t_field, t0, t1, every, "mean", index),
        "RH": efd_agg("lsst.sal.ESS.relativeHumidity", "relativeHumidityItem", t0, t1, every, "mean", index),
        "RH_max": efd_agg("lsst.sal.ESS.relativeHumidity", "relativeHumidityItem", t0, t1, every, "max", index),
        "Td": efd_agg("lsst.sal.ESS.dewPoint", "dewPointItem", t0, t1, every, "mean", index),
        "Td_max": efd_agg("lsst.sal.ESS.dewPoint", "dewPointItem", t0, t1, every, "max", index),
        "wind": efd_agg("lsst.sal.ESS.airFlow", "speed", t0, t1, every, "mean", index),
        "gust": efd_agg("lsst.sal.ESS.airFlow", "maxSpeed", t0, t1, every, "max", index),
    }
    w = pd.concat({k: v for k, v in cols.items()}, axis=1)
    if len(w) == 0:
        return w
    if w["Td"].isna().all() and not w["RH"].isna().all():
        # fall back to the page's approximation if no dewPoint topic for this index
        w["Td"] = w["T"] - (100 - w["RH"]) / 5
        w["Td_max"] = w["Td"]
    w["spread"] = w["T"] - w["Td"]
    # worst case within the bin: highest dew point against mean T
    w["spread_min"] = w["T"] - w["Td_max"]
    w["spread_formula"] = (100 - w["RH"]) / 5
    w["wind_avg5"] = w["wind"].rolling("5min").mean()
    return w


def precipitation(t0, t1):
    df = efd_raw("lsst.sal.ESS.logevent_precipitation", ["sensorName", "raining", "snowing", "salIndex"], t0, t1)
    if len(df) == 0:
        return df
    return df[(df.get("raining", False) == True) | (df.get("snowing", False) == True)]  # noqa: E712


def to_local(idx):
    return idx.tz_convert(SITE_TZ)

In [ ]:
def night_window(day_obs):
    sunset12, sunrise12 = rn_dayobs.day_obs_sunset_sunrise(day_obs, -12)
    sunset0, sunrise0 = rn_dayobs.day_obs_sunset_sunrise(day_obs, -0.833)
    utc = lambda t: pd.Timestamp(t.utc.datetime).tz_localize("UTC")  # noqa: E731
    return dict(sunset0=utc(sunset0), sunset12=utc(sunset12), sunrise12=utc(sunrise12), sunrise0=utc(sunrise0))


def _run_min(ok, minutes, step_min):
    # True where `ok` has been continuously True for at least `minutes`.
    n = max(int(round(minutes / step_min)), 1)
    return ok.astype(float).rolling(n, min_periods=n).min().fillna(0).astype(bool)


def evaluate_rules(w, win, g=G, step_min=1.0, precip_times=None):
    # Per-sample rule evaluation inside the night window. `w` is outside_weather() output on a regular grid.
    d = w.loc[(w.index >= win["sunset12"]) & (w.index <= win["sunrise12"])].copy()
    if len(d) == 0:
        return d
    wind_ok = (d["wind_avg5"].fillna(0) < g["wind_avg_close"]) & (d["gust"].fillna(0) < g["gust_close"])
    late = d.index > (win["sunrise0"] - pd.Timedelta(hours=g["before_sunrise_h"]))
    dry = pd.Series(True, index=d.index)
    if precip_times is not None and len(precip_times) > 0:
        last_wet = pd.Series(np.nan, index=d.index)
        for t in precip_times:
            last_wet[d.index >= t] = 1.0
        dry = last_wet.isna()  # conservative: any precipitation keeps the rest of the night closed
    n30 = max(int(round(g["reopen_stable_min"] / step_min)), 1)

    d["table_ok"] = (d["spread"] > g["spread_open"]) & wind_ok & dry
    d["table_close"] = (d["spread_min"] < g["spread_close"]) | ~wind_ok
    d["table_caution"] = (d["spread_min"] <= g["spread_open"]) & ~d["table_close"]
    d["prose_ok"] = (d["RH_max"] <= g["rh_prose"]) & wind_ok & dry
    d["precip_page_ok"] = (d["RH_max"] <= g["rh_precip"]) & wind_ok & dry

    # "May open" = criteria held for the stability window AND not within 1 h of sunrise
    d["open_literal_table"] = _run_min(d["table_ok"] & (d["spread_min"] > g["spread_open"]), g["reopen_stable_min"], step_min) & ~late
    d["open_literal_prose"] = _run_min(d["prose_ok"], g["reopen_stable_min"], step_min) & ~late
    d["open_literal_precip"] = _run_min(d["precip_page_ok"], 20, step_min) & ~late
    med = d["spread"].rolling(n30, min_periods=n30).median()
    mn = d["spread_min"].rolling(n30, min_periods=n30).min()
    d["open_proposed"] = (med > g["spread_open"]) & (mn > g["spread_close"]) & wind_ok & dry & ~late
    d["spike"] = d["RH_max"] > g["spike_rh"]
    return d


def latch_open(allowed):
    # Once the dome may open, it stays open until a closing limit (handled separately); here: first allowed time onward.
    a = allowed.astype(bool)
    return a.cummax() if a.any() else a


def hours(mask, step_min=1.0):
    return float(mask.sum()) * step_min / 60.0


def segments(mask):
    # Contiguous True runs -> DataFrame(start, end, minutes).
    m = mask.astype(bool)
    if not m.any():
        return pd.DataFrame(columns=["start", "end", "minutes"])
    grp = (m != m.shift()).cumsum()
    out = []
    for _, s in m[m].groupby(grp[m]):
        out.append((s.index[0], s.index[-1], (s.index[-1] - s.index[0]).total_seconds() / 60 + 1))
    return pd.DataFrame(out, columns=["start", "end", "minutes"])

## 1. One night in detail

In [ ]:
win = night_window(DAY_OBS)
t0 = win["sunset0"] - pd.Timedelta(hours=4)     # include the afternoon build-up and handover
t1 = win["sunrise0"] + pd.Timedelta(minutes=30)
print({k: to_local(pd.DatetimeIndex([v]))[0].strftime("%Y-%m-%d %H:%M %Z") for k, v in win.items()})

w = outside_weather(t0, t1, "1m")
print(f"outside samples: {len(w)}; columns: {list(w.columns)}")
wet = precipitation(t0, t1)
print(f"precipitation events (raining/snowing True): {len(wet)}")
display(w.describe().T.round(2))

In [ ]:
# Sanity check: sensors behind ESS:301 (outside weather station), and its dew point topic vs. the page's formula.
for topic in ["lsst.sal.ESS.temperature", "lsst.sal.ESS.relativeHumidity", "lsst.sal.ESS.dewPoint", "lsst.sal.ESS.airFlow"]:
    s = efd_raw(topic, ["sensorName", "location"], t0, t0 + pd.Timedelta(minutes=10), index=OUTSIDE_INDEX)
    print(topic, sorted(set(s.get("sensorName", []))), sorted(set(s.get("location", []))))
print("median |Td(topic) - Td(formula)| =", np.nanmedian(np.abs(w["spread"] - w["spread_formula"])).round(2), "C")

In [ ]:
r = evaluate_rules(w, win, G, 1.0, None if len(wet) == 0 else list(wet.index))
night_h = (win["sunrise12"] - win["sunset12"]).total_seconds() / 3600

def first_time(mask):
    return to_local(mask[mask].index[:1])[0].strftime("%H:%M") if mask.any() else "never"

rows = []
for col, label in [
    ("open_literal_prose", "Prose: RH <= 70% for 30 min"),
    ("open_literal_precip", "Precip page: RH <= 75% for 20 min"),
    ("open_literal_table", "Table, literal: spread > 5 C every minute for 30 min"),
    ("open_proposed", "Proposed: 30-min median spread > 5 C, min > 2 C"),
]:
    rows.append(dict(rule=label, first_allowed=first_time(r[col]),
                     hours_allowed=round(hours(r[col]), 2),
                     hours_if_opened_at_first=round(hours(latch_open(r[col]) & ~r["table_close"]), 2)))
summary = pd.DataFrame(rows)
print(f"Night (-12 to -12): {night_h:.2f} h")
display(summary)

print("Minutes in each table band during the night:")
display(pd.Series({
    "can open (spread > 5)": int(r["table_ok"].sum()),
    "caution (2-5)": int(r["table_caution"].sum()),
    "close (< 2 or wind)": int(r["table_close"].sum()),
}))

sp = segments(r["spike"])
if len(sp):
    sp["min_spread_C"] = [round(float(r.loc[a:b, "spread_min"].min()), 2) for a, b in zip(sp.start, sp.end)]
    sp["max_RH"] = [round(float(r.loc[a:b, "RH_max"].max()), 1) for a, b in zip(sp.start, sp.end)]
    sp["start"] = to_local(pd.DatetimeIndex(sp.start)).strftime("%H:%M")
    sp["end"] = to_local(pd.DatetimeIndex(sp.end)).strftime("%H:%M")
print(f"RH > {G['spike_rh']}% spikes: {len(sp)}")
display(sp)
print("Would any spike have forced a close had the dome been open?  ambient spread < 2 C minutes:",
      int((r["spread_min"] < G["spread_close"]).sum()))

### Inside / structure sensors

Every ESS index that publishes a dew point, compared with that index's first temperature channel. Pairing is per salIndex, so treat this as a screening list and check flagged sensors by name before quoting them. The guidance closes on any sensor within 2 °C of its dew point for about 20 min.

In [ ]:
dp_all = efd_raw("lsst.sal.ESS.dewPoint", ["dewPointItem", "sensorName", "location", "salIndex"], win["sunset12"], win["sunrise12"])
sens = []
if len(dp_all):
    for idx, grp in dp_all.groupby("salIndex"):
        td = grp["dewPointItem"].resample("1min").mean()
        tf = first_field("lsst.sal.ESS.temperature", "temperatureItem") or "temperatureItem0"
        tt = efd_agg("lsst.sal.ESS.temperature", tf, win["sunset12"], win["sunrise12"], "1m", "mean", idx)
        tt.index = tt.index.floor("1min")
        s = (tt - td.reindex(tt.index)).dropna()
        if len(s) == 0:
            continue
        below = s < G["sensor_close"]
        longest = segments(below)["minutes"].max() if below.any() else 0
        sens.append(dict(salIndex=int(idx), sensor=", ".join(sorted(set(grp.sensorName.astype(str))))[:60],
                         location=", ".join(sorted(set(grp.location.astype(str))))[:60],
                         min_spread_C=round(s.min(), 2), median_spread_C=round(s.median(), 2),
                         minutes_below_2C=int(below.sum()), longest_run_below_2C_min=float(longest)))
sens = pd.DataFrame(sens).sort_values("min_spread_C") if sens else pd.DataFrame()
display(sens)
if len(sens):
    trig = sens[sens.longest_run_below_2C_min >= G["sensor_close_min"]]
    print(f"Sensors meeting the 2 C / {G['sensor_close_min']} min close rule: {len(trig)}")

In [ ]:
# Dynalene at mirror level vs. outside dew point ("outside dew point 2 C lower than Dynalene" before opening;
# table: Dynalene - dew point > 0 C can open, below -2 C close).
dyn = efd_agg(DYN_TOPIC, DYN_FIELD, win["sunset12"], win["sunrise12"], "1m", "mean", client=cam) + DYN_OFFSET
if len(dyn):
    m = (dyn - r["Td"].reindex(dyn.index)).dropna()
    print(f"Dynalene(camera) - outside Td: min {m.min():.2f} C, median {m.median():.2f} C; "
          f"minutes < 2 C (pre-open check): {(m < 2).sum()}; minutes < -2 C (table close): {(m < -2).sum()}")
else:
    print(f"No {DYN_TOPIC}.{DYN_FIELD} in {CAMERA_DB} for this window; last query: {cam.last_query}")

In [ ]:
# Cloud proxy: DIMM and RINGSS measurement cadence (no measurements = no stars, or instrument closed).
# Neither has published since mid-2026 (DIMM last 2026-07-14, RINGSS 2026-06-15), so also read the allskycam
# cloud index (rubin_allskycam products) when they are reachable.
dimm = efd_raw("lsst.sal.DIMM.logevent_dimmMeasurement", ["fwhm", "salIndex"], win["sunset12"], win["sunrise12"])
ringss = efd_raw("lsst.sal.ESS.logevent_ringssMeasurement", "*", win["sunset12"], win["sunrise12"])
cad = pd.DataFrame({
    "dimm_per_10min": dimm["fwhm"].resample("10min").count() if len(dimm) else pd.Series(dtype=float),
    "ringss_per_10min": ringss.iloc[:, 0].resample("10min").count() if len(ringss) else pd.Series(dtype=float),
})
if len(dimm):
    print(f"DIMM median FWHM while measuring: {dimm['fwhm'].median():.2f} arcsec; first: {to_local(dimm.index[:1])[0]:%H:%M}")
display(cad.set_axis(to_local(cad.index)).T if len(cad) else cad)
import os
ALLSKY_OUT = os.environ.get("ALLSKY_OUT", "/sdf/data/rubin/user/stalder/allsky")
cam_path = os.path.join(ALLSKY_OUT, "cloud", f"{DAY_OBS}.parquet")
sky = pd.DataFrame()
if os.path.exists(cam_path):
    sky = pd.read_parquet(cam_path).query("geometry_ok")
    sky.index = pd.DatetimeIndex(Time(sky["mjd"].to_numpy(), format="mjd").utc.datetime).tz_localize("UTC")
    sky = sky.loc[(sky.index >= win["sunset12"]) & (sky.index <= win["sunrise12"])]
    cloud10 = sky["cloud_idx"].resample("10min").median()
    print(f"allskycam frames in the night: {len(sky)}; fraction with cloud index > 0.3: {(sky.cloud_idx > 0.3).mean():.2f}, > 0.5: {(sky.cloud_idx > 0.5).mean():.2f}")
    display(cloud10.set_axis(to_local(cloud10.index)).round(2).to_frame("allsky cloud_idx (10-min median)").T)
else:
    print(f"No allskycam cloud table at {cam_path}")


In [ ]:
# What the system recorded: dome, observatory state, Watcher, narrative log, scripts.
dome = observatory_status.get_dome_open_close(Time(t0), Time(t1), efd)
display(dome.query("day_obs == @DAY_OBS") if "day_obs" in dome else dome)

try:
    states = observatory_status.get_observatory_state_times(Time(t0), Time(t1), efd)
    display(states)
except Exception as e:  # noqa: BLE001
    print(f"observatory state: {e}")

alarms = efd_raw("lsst.sal.Watcher.logevent_alarm", ['"name"', "severity", "reason"], t0, t1)  # name is an InfluxQL keyword
pat = re.compile(r"humid|dew|weather|wind|rain|precip|ESS:|dome", re.I)
if len(alarms):
    alarms = alarms[alarms["name"].astype(str).str.contains(pat)].sort_index()
    # one row per episode: severity > 1 events of the same alarm, merged across gaps < 10 min (severity flickers)
    ep = []
    for name, g in alarms[alarms["severity"] > 1].groupby("name"):
        brk = g.index.to_series().diff() > pd.Timedelta(minutes=10)
        for _, e in g.groupby(brk.cumsum()):
            ep.append(dict(name=name, start=e.index[0], end=e.index[-1], max_severity=int(e.severity.max()),
                           minutes=round((e.index[-1] - e.index[0]).total_seconds() / 60, 1), first_reason=e.reason.iloc[0][:110]))
    alarms = pd.DataFrame(ep)
    if len(alarms):
        alarms = alarms.sort_values("start")
        alarms["start"] = to_local(pd.DatetimeIndex(alarms.start)).strftime("%m-%d %H:%M")
        alarms["end"] = to_local(pd.DatetimeIndex(alarms.end)).strftime("%m-%d %H:%M")
        alarms = alarms.reset_index(drop=True)
display(alarms)

nlog = endpoints["narrative_log"].query_log(Time(t0), Time(t1))
if len(nlog):
    nlog = nlog[~nlog["component"].astype(str).str.contains("AuxTel")]
    nlog.index = to_local(nlog.index)
    cols = [c for c in ["component", "user_id", "time_lost", "time_lost_type", "message_text"] if c in nlog]
    display(nlog[cols])

In [ ]:
try:
    import rubin_nights.scriptqueue as scriptqueue
    scripts = scriptqueue.get_script_stream(Time(t0), Time(t1), efd)
    scripts.index = to_local(pd.DatetimeIndex(scripts.index)) if isinstance(scripts.index, pd.DatetimeIndex) else scripts.index
    keep = [c for c in scripts.columns if re.search(r"script|block|config|salIndex", c, re.I)][:6]
    display(scripts[keep] if keep else scripts.head(50))
except Exception as e:  # noqa: BLE001
    print(f"script stream: {e}")

In [ ]:
# Figure: one night against the limits.
L = lambda s: s.set_axis(to_local(s.index))  # noqa: E731
fig, ax = plt.subplots(4, 1, figsize=(11, 10), sharex=True, gridspec_kw=dict(height_ratios=[2, 2, 1.3, 1.1]))
xw = L(w)

ax[0].plot(xw.index, xw["T"], label="T (ESS:301)")
ax[0].plot(xw.index, xw["Td"], label="dew point (ESS:301)")
if len(dyn):
    ax[0].plot(L(dyn).index, L(dyn), label="Dynalene (camera supply)")
ax[0].set_ylabel("°C"); ax[0].legend(loc="upper right", fontsize=8)

ax[1].plot(xw.index, xw["spread"], color="k", lw=1, label="spread (1-min mean)")
ax[1].plot(xw.index, xw["spread_min"], color="0.6", lw=0.8, label="spread (1-min worst)")
ax[1].axhspan(-5, G["spread_close"], color="tab:red", alpha=0.12)
ax[1].axhspan(G["spread_close"], G["spread_open"], color="tab:orange", alpha=0.12)
ax[1].set_ylim(bottom=max(-1, np.nanmin(xw["spread_min"]) - 1))
ax[1].set_ylabel("T − T_dew (°C)")
a2 = ax[1].twinx()
a2.plot(xw.index, xw["RH"], color="tab:blue", lw=0.8, alpha=0.7)
for v, ls in [(G["rh_prose"], ":"), (G["rh_precip"], "--"), (G["spike_rh"], "-")]:
    a2.axhline(v, color="tab:blue", ls=ls, lw=0.8)
a2.set_ylabel("RH (%)", color="tab:blue")
ax[1].legend(loc="upper left", fontsize=8)

ax[2].plot(xw.index, xw["wind_avg5"], label="wind, 5-min avg")
ax[2].plot(xw.index, xw["gust"], lw=0.6, label="gust")
ax[2].axhline(G["wind_avg_close"], color="tab:red", ls="--", lw=0.8)
ax[2].axhline(G["gust_close"], color="tab:red", lw=0.8)
ax[2].set_ylabel("m/s"); ax[2].legend(loc="upper right", fontsize=8)

lanes = [("open_proposed", "proposed"), ("open_literal_table", "table, literal"),
         ("open_literal_precip", "precip page"), ("open_literal_prose", "prose RH 70%")]
for i, (col, lab) in enumerate(lanes):
    m = L(r[col])
    for _, sgm in segments(m).iterrows():
        ax[3].broken_barh([(mdates.date2num(sgm.start), (sgm.minutes) / 1440)], (i - 0.35, 0.7), color="tab:green")
    ax[3].text(1.002, i, lab, transform=ax[3].get_yaxis_transform(), va="center", fontsize=8)
if "open_time" in dome:
    for _, dr in dome.query("day_obs == @DAY_OBS").iterrows():
        if pd.notna(dr.open_time):
            o = to_local(pd.DatetimeIndex([dr.open_time]))[0]
            c = to_local(pd.DatetimeIndex([dr.close_time]))[0] if pd.notna(dr.close_time) else xw.index[-1]
            ax[3].broken_barh([(mdates.date2num(o), mdates.date2num(c) - mdates.date2num(o))], (len(lanes) - 0.35, 0.7), color="k")
ax[3].text(1.002, len(lanes), "dome actually open", transform=ax[3].get_yaxis_transform(), va="center", fontsize=8)
ax[3].set_yticks([]); ax[3].set_ylim(-0.6, len(lanes) + 0.6)
ax[3].set_title("When each rule would allow opening (green)", fontsize=9, loc="left")

for a in ax:
    for k in ["sunset12", "sunrise12"]:
        a.axvline(to_local(pd.DatetimeIndex([win[k]]))[0], color="0.3", ls=":", lw=0.8)
    if len(nlog):
        for t in nlog.index:
            a.axvline(t, color="tab:purple", alpha=0.25, lw=0.6)
ax[-1].xaxis.set_major_formatter(mdates.DateFormatter("%H:%M", tz=SITE_TZ))
fig.suptitle(f"dayObs {DAY_OBS}: outside conditions vs. Simonyi open/close guidance (purple = narrative log)")
fig.tight_layout()
plt.show()

## 2. Commissioning through operations

Each night is scored on a 5-min grid against the same rules. For each night, `allowed_*` is the hours the rule would have permitted opening, and `open_h` is the hours the dome was actually open. These are categories to triage, not verdicts:

* **possible_missed:** dome closed (open_h < 0.25) while the proposed rule allowed ≥ 2 h. Check the narrative log and the closure list: engineering, faults, staffing or road closures may explain it.
* **open_past_limit:** minutes the dome was open while the ambient spread was < 2 °C or wind was over a limit.

Clouds and precipitation forecasts are not modeled, so this scan finds candidates only.

In [ ]:
def season_scan(day_min, day_max, every=SEASON_BIN):
    step = pd.Timedelta(every).total_seconds() / 60
    t_lo = rn_dayobs.day_obs_to_time(day_min)
    t_hi = rn_dayobs.day_obs_to_time(day_max) + TimeDelta(1, format="jd")
    chunks, t = [], t_lo
    while t < t_hi:
        tn = min(t + TimeDelta(14, format="jd"), t_hi)
        chunks.append(outside_weather(t, tn, every))
        t = tn
    W = pd.concat([c for c in chunks if len(c)]).sort_index()
    W = W[~W.index.duplicated()]
    W["wind_avg5"] = W["wind"]  # at 5-min binning the mean is already the 5-min average
    dome_all = observatory_status.get_dome_open_close(t_lo, t_hi, efd)
    nlog_all = endpoints["narrative_log"].query_log(t_lo, t_hi, {"min_time_lost": "0.00001"})
    if len(nlog_all):
        nlog_all = nlog_all[~nlog_all["component"].astype(str).str.contains("AuxTel")]
    out = []
    for day in rn_dayobs.day_obs_list(t_lo, t_hi - TimeDelta(1, format="jd")):
        day = int(str(day).replace("-", ""))
        wn = night_window(day)
        rr = evaluate_rules(W, wn, G, step)
        if len(rr) == 0:
            continue
        dd = dome_all.query("day_obs == @day") if "day_obs" in dome_all else pd.DataFrame()
        open_mask = pd.Series(False, index=rr.index)
        for _, x in dd.iterrows():
            if pd.notna(x.get("open_time")):
                o = pd.Timestamp(x.open_time); o = o.tz_localize("UTC") if o.tzinfo is None else o
                c = pd.Timestamp(x.close_time) if pd.notna(x.get("close_time")) else rr.index[-1]
                c = c.tz_localize("UTC") if c.tzinfo is None else c
                open_mask |= (rr.index >= o) & (rr.index <= c)
        lost = nlog_all[(nlog_all.index >= wn["sunset12"] - pd.Timedelta(hours=3)) & (nlog_all.index <= wn["sunrise12"] + pd.Timedelta(hours=3))] if len(nlog_all) else pd.DataFrame()
        out.append(dict(
            day_obs=day,
            night_h=round((wn["sunrise12"] - wn["sunset12"]).total_seconds() / 3600, 2),
            open_h=round(hours(open_mask, step), 2),
            allowed_prose_h=round(hours(rr["open_literal_prose"], step), 2),
            allowed_table_literal_h=round(hours(rr["open_literal_table"], step), 2),
            allowed_proposed_h=round(hours(rr["open_proposed"], step), 2),
            open_past_limit_min=int((open_mask & rr["table_close"]).sum() * step),
            spikes=int(len(segments(rr["spike"]))),
            min_spread_C=round(float(rr["spread_min"].min()), 1),
            max_gust=round(float(rr["gust"].max()), 1),
            log_weather_h=round(float(lost.query("time_lost_type == 'weather'")["time_lost"].sum()), 2) if len(lost) else 0.0,
            log_fault_h=round(float(lost.query("time_lost_type == 'fault'")["time_lost"].sum()), 2) if len(lost) else 0.0,
        ))
    S = pd.DataFrame(out).set_index("day_obs")
    S["possible_missed"] = (S.open_h < 0.25) & (S.allowed_proposed_h >= 2)
    S["prose_vs_proposed_h"] = (S.allowed_proposed_h - S.allowed_prose_h).round(2)
    return S


if RUN_SEASON:
    S = season_scan(SEASON_MIN, SEASON_MAX)
    S.to_csv(f"dome_guidance_season_{SEASON_MIN}_{SEASON_MAX}.csv")
    print(f"{len(S)} nights; possible_missed: {S.possible_missed.sum()}; nights open past a limit: {(S.open_past_limit_min > 0).sum()}")
    print("Hours allowed, summed: prose 70% = %.0f, table literal = %.0f, proposed = %.0f; actual open = %.0f"
          % (S.allowed_prose_h.sum(), S.allowed_table_literal_h.sum(), S.allowed_proposed_h.sum(), S.open_h.sum()))
    display(S.query("possible_missed or open_past_limit_min > 0"))

In [ ]:
if RUN_SEASON:
    m = S.copy()
    m.index = pd.to_datetime(m.index.astype(str))
    mo = m.resample("MS")[["night_h", "open_h", "allowed_prose_h", "allowed_proposed_h"]].sum()
    fig, ax = plt.subplots(figsize=(11, 4))
    x = np.arange(len(mo)); wdt = 0.27
    ax.bar(x - wdt, mo.allowed_proposed_h, wdt, label="allowed, proposed rule")
    ax.bar(x, mo.allowed_prose_h, wdt, label="allowed, RH 70% prose")
    ax.bar(x + wdt, mo.open_h, wdt, label="dome actually open", color="k")
    ax.set_xticks(x); ax.set_xticklabels(mo.index.strftime("%Y-%m"), rotation=60)
    ax.set_ylabel("hours (−12° to −12°)"); ax.legend(); ax.set_title("Night hours allowed by each rule vs. hours open, by month")
    fig.tight_layout(); plt.show()